# Evaluate Models on external hopkins data

- Import cleaned raw data
- Transform on ML pipeline
- Pass through models

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.data_utils import get_data, get_models
from src.eval import evaluate_models, NumpyEncoder, BIN_NAMES, get_bin_row
import pandas as pd
import json
from shutil import rmtree

print(f"Path: {BASE_PATH}")

Globals

In [ ]:
DATA_DICT = {
    "base": get_data(is_nomo=False),
    "nomo": get_data(is_nomo=True),
}


# Models
model_dir = BASE_PATH / "artifacts/models/trained"
model_prefix_list = ["lgbm", "xgb", "knn", "svc", "nn", "stack"]
base_model_dict = {}

## Base models
base_model_dict = get_models(model_prefix_list, model_dir)
## Nomogram
nomo_model_dict = get_models(["lr"], model_dir)

assert DATA_DICT["base"]["X"].isna().sum().sum() == 0
assert DATA_DICT["nomo"]["X"].isna().sum().sum() == 0

In [ ]:
THRESHOLD_DICT = {
    "lgbm": 0.36,
    "xgb": 0.847,
    "knn": 0.674,
    "svc": 0.56,
    "nn": 0.607,
    "stack": 0.575,
    "lr": 0.371,
}

## Evaluate

In [ ]:
all_models_test_dict = {}
ALL_DICT = {}
cls_rows = []
cls_index = []
n_bootstraps = 5000
save_path = BASE_PATH / "results"

Base

In [ ]:
## BASE MODELS
# set n bins in src/eval.py
class_report_dict, bin_report_dict = evaluate_models(
    model_dict=base_model_dict,
    X=DATA_DICT["base"]["X"],
    y=DATA_DICT["base"]["y"].values.ravel(),
    threshold_dict=THRESHOLD_DICT,
    results_path=save_path,
    bin_import_dir=BASE_PATH / "artifacts/bin_thresholds",
    show_cm=False,
    show_roc=False,
    show_cal=False,
    n_bootstraps=n_bootstraps,
    show_progress=False,
)
## ONLY export test (have access to train/val if need be)
for model, metrics in class_report_dict.items():
    cls_rows.append(metrics)
    cls_index.append(model)

Nomogram

In [ ]:
## Nomogram (LR) MODEL
nomo_class_report_dict, nomo_bin_report_dict = evaluate_models(
    model_dict=nomo_model_dict,
    X=DATA_DICT["nomo"]["X"],
    y=DATA_DICT["nomo"]["y"].values.ravel(),
    threshold_dict=THRESHOLD_DICT,
    results_path=save_path,
    bin_import_dir=BASE_PATH / "artifacts/bin_thresholds",
    show_cm=False,
    show_roc=False,
    show_cal=False,
    n_bootstraps=n_bootstraps,
    show_progress=False,
)
# Class
ALL_DICT["class"] = class_report_dict
ALL_DICT["class"]["lr"] = nomo_class_report_dict
# Bins
bin_report_dict["lr"] = nomo_bin_report_dict["lr"]
ALL_DICT["bins"] = bin_report_dict

for model, metrics in nomo_class_report_dict.items():
    cls_rows.append(metrics)
    cls_index.append(model)

Save all results in JSON

In [ ]:
all_models_outcomes_df = pd.DataFrame(cls_rows, index=cls_index)
all_save_path = save_path / "tables" / "all_dict_results.json"
if all_save_path.exists():
    all_save_path.unlink()
all_save_path.parent.mkdir(exist_ok=True, parents=True)
with open(all_save_path, "w") as f:
    json.dump(ALL_DICT, f, cls=NumpyEncoder, indent=2)

Re-format bins

In [ ]:
# Load it back
with open(all_save_path, "r") as f:
    loaded_dict = json.load(f)
bin_rows = []
bin_index = []
bins_dict = loaded_dict["bins"]
for model_name, bins in bins_dict.items():
    for bin_name, metrics in bins.items():
        row = get_bin_row(metrics)
        bin_rows.append(row)
        bin_index.append((model_name, bin_name))
# Create DataFrame with MultiIndex
df = pd.DataFrame(
    bin_rows, index=pd.MultiIndex.from_tuples(bin_index, names=["Model", "Bin"])
)

# Convert Bin level to categorical with custom order
df.index = df.index.set_levels(  # type: ignore
    pd.CategoricalIndex(df.index.levels[1], categories=BIN_NAMES, ordered=True),  # type: ignore
    level=1,
)

# Sort with the custom ordering
df = df.sort_index()

# Convert to flat table
df_flat = df.reset_index()
df_flat

Export tables

In [ ]:
report_path = save_path / "tables" / "metrics"
if report_path.exists():
    rmtree(report_path)
report_path.mkdir(exist_ok=True, parents=True)
df_flat.to_excel(report_path / "bin_report.xlsx")
all_models_outcomes_df.to_excel(report_path / "class_report.xlsx")